# Exp 04: Dynamic Shape Fix

目标：
1. 修复 Exp03 中的 dynamic shape recompile 问题。
2. 对比 `dynamic=False`、automatic dynamic、`mark_dynamic`、unbacked symbol、`drop_last`、bucketing 的效果。
3. 理解每种方案的工程适用场景和代价。

推荐脚本版配合日志运行：

```bash
TORCH_LOGS=dynamic python experiments_py/exp04_dynamic_shape_fix.py
TORCH_LOGS=recompiles python experiments_py/exp04_dynamic_shape_fix.py
```

如果 `TORCH_LOGS=recompiles` 没有输出，通常说明没有发生新的 recompile。

## 方法总览

- `mark_dynamic(x, dim, min, max)`：显式告诉 Dynamo 某个维度是动态的，第一次调用就编译 dynamic graph。
- automatic dynamic：默认策略。第一次按静态 shape 编译，遇到不同 shape 后再升级为 dynamic。
- unbacked symbol：让 automatic dynamic 不依赖初始 concrete value，可绕开部分 `0/1 specialization` 问题。
- `drop_last=True`：DataLoader 侧避免最后一个过小 batch，是最便宜的工程防线。
- bucketing：把无限多 shape pad 到有限桶，常用于 NLP sequence length。

In [ ]:
import contextlib
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch._dynamo.utils import counters

assert torch.cuda.is_available(), "This notebook requires a CUDA GPU."

DEVICE = "cuda"
DTYPE = torch.bfloat16
IN_DIM = 512
OUT_DIM = 128

BATCH_SEQUENCE = [32] * 8 + [16] * 6 + [8] * 4 + [24] * 6 + [32] * 8 + [7]

torch.manual_seed(42)

In [ ]:
class SimpleNet(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc1 = nn.Linear(IN_DIM, 1024)
        self.fc2 = nn.Linear(1024, OUT_DIM)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(F.gelu(self.fc1(x)))


def make_model() -> SimpleNet:
    return SimpleNet().to(DEVICE, dtype=DTYPE)


def clear_grads(model: nn.Module) -> None:
    for param in model.parameters():
        param.grad = None


@contextlib.contextmanager
def temporary_unbacked_symbols():
    import torch._dynamo.config as dynamo_config

    old_value = getattr(dynamo_config, "automatic_dynamic_shapes_mark_as", None)
    dynamo_config.automatic_dynamic_shapes_mark_as = "unbacked"
    try:
        yield
    finally:
        if old_value is not None:
            dynamo_config.automatic_dynamic_shapes_mark_as = old_value


def run_batches(
    model: nn.Module,
    batches: list[int],
    *,
    mark_dynamic: bool = False,
) -> tuple[int, float]:
    counters.clear()
    start = time.perf_counter()

    for batch_size in batches:
        x = torch.randn(batch_size, IN_DIM, device=DEVICE, dtype=DTYPE)
        y = torch.randn(batch_size, OUT_DIM, device=DEVICE, dtype=DTYPE)

        if mark_dynamic:
            torch._dynamo.mark_dynamic(x, 0, min=2, max=512)
            torch._dynamo.mark_dynamic(y, 0, min=2, max=512)

        loss = F.mse_loss(model(x), y)
        loss.backward()
        clear_grads(model)

    torch.cuda.synchronize()
    elapsed_ms = (time.perf_counter() - start) * 1000
    return counters["stats"]["unique_graphs"], elapsed_ms

## 对照组：`dynamic=False`

先保留 Exp03 的问题设置。`dynamic=False` 会关闭 automatic dynamic，所以不同 batch size 更容易对应多个独立 graph。

In [ ]:
results = {}

static_model = torch.compile(make_model(), mode="default", fullgraph=False, dynamic=False)
ug_static, ms_static = run_batches(static_model, BATCH_SEQUENCE)
results["dynamic=False"] = (ug_static, ms_static)

print(f"batch sequence length: {len(BATCH_SEQUENCE)}")
print(f"unique_graphs={ug_static}  total_time={ms_static:.0f} ms")

## 方案 A：`mark_dynamic`

`mark_dynamic` 是最直接的修复方式。它必须在 compiled function 外部、每次传入新 tensor 后调用。

这里用 `min=2` 是为了明确告诉编译器 batch 维不会是 0 或 1，从而避开 `0/1 specialization` 的常见坑。

In [ ]:
mark_model = torch.compile(make_model(), mode="default", fullgraph=False, dynamic=None)
ug_mark, ms_mark = run_batches(mark_model, BATCH_SEQUENCE, mark_dynamic=True)
results["mark_dynamic"] = (ug_mark, ms_mark)

print(f"unique_graphs={ug_mark}  total_time={ms_mark:.0f} ms")
print("第一次调用就按 dynamic batch 维编译，后续 batch 变化应尽量复用。")

## 方案 B：automatic dynamic

`dynamic=None` 是默认行为：第一次按静态 shape 编译；当后续发现同一维度变化，Dynamo 会尝试把该维升级为 symbolic dynamic shape。

优点是零代码改动；缺点是通常会浪费一次 static 编译。

In [ ]:
auto_model = torch.compile(make_model(), mode="default", fullgraph=False, dynamic=None)
ug_auto, ms_auto = run_batches(auto_model, BATCH_SEQUENCE)
results["automatic dynamic"] = (ug_auto, ms_auto)

print(f"unique_graphs={ug_auto}  total_time={ms_auto:.0f} ms")

## 方案 C：unbacked symbol

`automatic_dynamic_shapes_mark_as = "unbacked"` 是更进阶的配置。它让 automatic dynamic 使用不绑定初始 concrete value 的 symbolic 维度，能绕开部分 `0/1 specialization` 问题。

它也有代价：如果模型里写了 `if batch == 1:` 这类依赖 batch 具体值的 Python 分支，可能更容易 graph break。

In [ ]:
with temporary_unbacked_symbols():
    unbacked_model = torch.compile(make_model(), mode="default", fullgraph=False, dynamic=None)
    ug_unbacked, ms_unbacked = run_batches(unbacked_model, BATCH_SEQUENCE)

results["unbacked"] = (ug_unbacked, ms_unbacked)
print(f"unique_graphs={ug_unbacked}  total_time={ms_unbacked:.0f} ms")

## 方案 D/E：`drop_last` 与 bucketing

`drop_last=True` 是 DataLoader 侧的简单修复：不要产生最后那个过小 batch。

Bucketing 则是把任意 batch size 或 sequence length pad 到有限几个桶。它用一点额外计算换来更少的 shape 种类，常用于 NLP 的动态 sequence length。

In [ ]:
drop_last_batches = [batch_size for batch_size in BATCH_SEQUENCE if batch_size >= 8]
drop_model = torch.compile(make_model(), mode="default", fullgraph=False, dynamic=None)
ug_drop, ms_drop = run_batches(drop_model, drop_last_batches, mark_dynamic=True)
results["drop_last + mark_dynamic"] = (ug_drop, ms_drop)

BUCKETS = [8, 16, 32, 64]


def bucket_size(batch_size: int) -> int:
    for bucket in BUCKETS:
        if batch_size <= bucket:
            return bucket
    return BUCKETS[-1]


bucket_model = torch.compile(make_model(), mode="default", fullgraph=False, dynamic=False)
counters.clear()
start = time.perf_counter()

for batch_size in BATCH_SEQUENCE:
    bucket = bucket_size(batch_size)
    x = torch.zeros(bucket, IN_DIM, device=DEVICE, dtype=DTYPE)
    y = torch.zeros(bucket, OUT_DIM, device=DEVICE, dtype=DTYPE)
    x[:batch_size] = torch.randn(batch_size, IN_DIM, device=DEVICE, dtype=DTYPE)
    y[:batch_size] = torch.randn(batch_size, OUT_DIM, device=DEVICE, dtype=DTYPE)

    loss = F.mse_loss(bucket_model(x), y)
    loss.backward()
    clear_grads(bucket_model)

torch.cuda.synchronize()
ug_bucket = counters["stats"]["unique_graphs"]
ms_bucket = (time.perf_counter() - start) * 1000
results["bucketing"] = (ug_bucket, ms_bucket)

print(f"drop_last + mark_dynamic: unique_graphs={ug_drop}, total_time={ms_drop:.0f} ms")
print(f"bucketing {BUCKETS}: unique_graphs={ug_bucket}, total_time={ms_bucket:.0f} ms")

In [ ]:
print(f"{'strategy':28s} {'unique_graphs':>14s} {'total ms':>12s}")
print("-" * 58)
for name, (unique_graphs, elapsed_ms) in results.items():
    print(f"{name:28s} {unique_graphs:14d} {elapsed_ms:12.0f}")

## 小结

推荐顺序：

1. 训练 DataLoader 优先设置 `drop_last=True`，避免 batch 维为 1。
2. 对明确会变化的维度使用 `mark_dynamic(..., min=2, max=...)`。
3. 不想改代码时先依赖 automatic dynamic，但要接受一次额外编译。
4. 需要处理 batch=1/0 时再评估 unbacked symbol。
5. sequence length 高度变化时使用 bucketing，把无限 shape 变成有限桶。

下一节进入控制流：哪些 `if` 可以被 Dynamo 处理，哪些会导致 graph break。